# Evaluate Performance Using SVM Regression (RBF Kernel) 

In [34]:
# Import statements (add as needed)

import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, cross_validate, train_test_split
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, FunctionTransformer, PowerTransformer
from sklearn.impute import SimpleImputer

from sklearn.compose import (
    TransformedTargetRegressor,
    make_column_transformer,
)

from sklearn.pipeline import make_pipeline
from skopt import BayesSearchCV
from missforest import MissForest

from sklearn.multioutput import MultiOutputRegressor
from sklearn.multioutput import RegressorChain
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor

In [ ]:
# load both feature engineered dataset.

fe_df = pd.read_csv('../data/feature_engineered_training_set.csv')

In [3]:
fe_df = fe_df.drop(columns=['Unnamed: 0'])
fe_df.head()

,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,elevation,EVI,NDVI,Land Surface Temperature,nir,green,...,total_precipitation_sum,volumetric_soil_water,precipitation,cec_pH_interaction,phosphorous_pH_interaction,NDVI_LST_interaction,flow_acc_clay_interaction,flow_acc_phosphorous_interaction,evaporation_precipitation_ratio,cec_clay_ratio
0,128.912,555.0,10.0,174.2,167.155040,241.0,636.0,15845.0,11190.0,11426.0,...,0.000001,0.008799,0.488024,1800.0,1725.0,10077420.0,8.676030e+07,9.502319e+07,-37.216627,1.142857
1,74.720,162.9,163.0,124.1,1521.251493,4643.0,7656.0,15190.0,17658.5,9550.0,...,0.007830,0.458929,97.342756,1755.0,1365.0,116294640.0,3.369889e+05,2.440265e+05,-0.523901,0.931034
2,89.254,573.0,80.0,127.5,1471.379902,3984.0,6276.0,14879.0,15210.0,10720.0,...,0.006964,0.393498,96.911494,1472.0,1344.0,93380604.0,2.800000e+01,2.100000e+01,-0.575803,0.821429
3,82.000,203.6,101.0,129.7,1342.659998,2217.0,3957.0,15228.5,14887.0,10943.0,...,0.007228,0.409018,102.307634,1495.0,1430.0,60259174.5,4.884100e+05,4.132700e+05,-0.538889,0.884615
4,56.100,145.1,151.0,129.2,1355.983661,4136.0,7396.0,15130.0,16828.5,9502.5,...,0.004980,0.437815,96.513882,1600.0,1280.0,111901480.0,1.337840e+05,9.556000e+04,-0.784851,0.892857


**SVM is quite sensitive to scaling, hence scale with standard scaler. Lets see if feature engineering helped during feature selection**

In [4]:
train_df, test_df = train_test_split(fe_df, test_size=0.3)      # create train set and test set to eval performance

In [5]:
train_df.head()

,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,pet,elevation,EVI,NDVI,Land Surface Temperature,nir,green,...,total_precipitation_sum,volumetric_soil_water,precipitation,cec_pH_interaction,phosphorous_pH_interaction,NDVI_LST_interaction,flow_acc_clay_interaction,flow_acc_phosphorous_interaction,evaporation_precipitation_ratio,cec_clay_ratio
6709,180.469,528.00,20.0,248.30000,305.502681,767.5,1254.5,15258.5,9801.5,9961.5,...,0.000001,0.023224,0.000000,1606.0,1533.0,1.914179e+07,315.000000,315.000000,-13.640771,1.047619
4754,209.156,1056.45,184.0,187.40001,1252.920472,2178.0,3693.0,15533.0,14829.5,10139.5,...,0.000041,0.232401,43.544638,1518.0,1320.0,5.736337e+07,812249.418605,649799.534884,-47.878838,0.920000
7640,91.340,401.00,20.0,165.00000,827.817958,4577.0,7175.0,14957.0,16977.0,8880.5,...,0.001658,0.398513,126.332161,1176.0,1288.0,1.073165e+08,111129.289474,82450.763158,-1.944391,0.677419
6875,123.278,340.50,23.0,158.60000,1560.779641,1363.5,2916.0,15228.0,12535.5,10114.0,...,0.000390,0.307589,53.536292,1690.0,1300.0,4.440485e+07,969493.000000,692495.000000,-3.824146,0.928571
6970,10.000,61.40,20.0,172.20000,1141.929640,2445.0,4066.0,15389.5,16524.5,10678.0,...,0.002143,0.375245,15.038089,1159.0,1220.0,6.257371e+07,47.923077,45.641026,-1.342796,0.904762


In [6]:
train_df.shape

(6523, 33)

### Preprocess Data (Scale + Impute)

In [10]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6523 entries, 6709 to 3602
Data columns (total 33 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Total Alkalinity                  6523 non-null   float64
 1   Electrical Conductance            6523 non-null   float64
 2   Dissolved Reactive Phosphorus     6523 non-null   float64
 3   pet                               6523 non-null   float64
 4   elevation                         6523 non-null   float64
 5   EVI                               6523 non-null   float64
 6   NDVI                              6523 non-null   float64
 7   Land Surface Temperature          6523 non-null   float64
 8   nir                               5749 non-null   float64
 9   green                             5749 non-null   float64
 10  swir16                            5749 non-null   float64
 11  swir22                            5749 non-null   float64
 12  NDMI    

In [20]:
train_df.columns[train_df.isnull().any()].tolist()

['nir',
 'green',
 'swir16',
 'swir22',
 'NDMI',
 'MNDWI',
 'skin_temperature',
 'soil_temperature',
 'temperature_2m',
 'total_evaporation_sum',
 'total_precipitation_sum',
 'volumetric_soil_water',
 'precipitation',
 'evaporation_precipitation_ratio']

Preprocessing must handle missing values, then scale all data. Can be done in a pipeline. Scaling for all features, imputation only for those with missing values. 

In [ ]:
## Data to be imputed with missForest imputation, scaled with standard scaler. 

imputing_feats = ['nir',
 'green',
 'swir16',
 'swir22',
 'NDMI',
 'MNDWI',
 'skin_temperature',
 'soil_temperature',
 'temperature_2m',
 'total_evaporation_sum',
 'total_precipitation_sum',
 'volumetric_soil_water',
 'precipitation',
 'evaporation_precipitation_ratio']


In [21]:
# Impute train data using MissForest

mf = MissForest()
mf.fit(train_df)
train_imp = mf.transform(train_df)
train_imp

/Users/rohan/Desktop/EY-Data-And-AI-Challenge/EY-AI-And-Data-Challenge/venv/lib/python3.12/site-packages/missforest/missforest.py:333: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "
 80%|████████  | 4/5 [00:40<00:10, 10.10s/it]/Users/rohan/Desktop/EY-Data-And-AI-Challenge/EY-AI-And-Data-Challenge/venv/lib/python3.12/site-packages/missforest/missforest.py:303: UserWarning: NRMSE increased.
  warnings.warn("NRMSE increased.")
/Users/rohan/Desktop/EY-Data-And-AI-Challenge/EY-AI-And-Data-Challenge/venv/lib/python3.12/site-packages/missforest/missforest.py:453: UserWarning: Stopping criterion triggered during fitting. Before last imputation matrix will be returned.
  warnings.warn(
 80%|████████  | 4/5 [00:50<00:12, 12.63s/it]
/Users/rohan/Desktop/EY-Data-And-AI-Challenge/EY-AI-And-Data-Challenge/venv/lib/python3.12/site-packages/missfo

,Total Alkalinity,flow_acc_phosphorous_interaction,flow_acc_clay_interaction,NDVI_LST_interaction,phosphorous_pH_interaction,cec_pH_interaction,flow_accumulation,phosphorous,clay,cec,...,soil_temperature,skin_temperature,evaporation_precipitation_ratio,total_evaporation_sum,swir16,nir,green,MNDWI,NDMI,swir22
6709,180.4690,3.150000e+02,3.150000e+02,1.914179e+07,1533.0,1606.0,15.000000,21.0,21.0,22.0,...,293.064711,292.979788,-13.640771,-0.000028,9245.500000,9801.50000,9961.500000,0.037278,0.029191,8826.500000
4754,209.1560,6.497995e+05,8.122494e+05,5.736337e+07,1320.0,1518.0,32489.976744,20.0,25.0,23.0,...,294.115392,293.622035,-47.878838,-0.002033,14071.000000,14829.50000,10139.500000,-0.162388,0.026245,11931.500000
7640,91.3400,8.245076e+04,1.111293e+05,1.073165e+08,1288.0,1176.0,3584.815789,23.0,31.0,21.0,...,292.685397,292.207076,-1.944391,-0.003225,12252.000000,16977.00000,8880.500000,-0.159541,0.161655,9523.500000
6875,123.2780,6.924950e+05,9.694930e+05,4.440485e+07,1300.0,1690.0,34624.750000,20.0,28.0,26.0,...,294.872040,294.868462,-3.824146,-0.001494,14015.000000,12535.50000,10114.000000,-0.161673,-0.055724,11770.000000
6970,10.0000,4.564103e+01,4.792308e+01,6.257371e+07,1220.0,1159.0,2.282051,20.0,21.0,19.0,...,296.790461,295.917322,-1.342796,-0.002879,18903.500000,16524.50000,10678.000000,-0.278062,-0.067150,16289.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1169,306.3915,6.497995e+05,8.122494e+05,5.954559e+07,1320.0,1518.0,32489.976744,20.0,25.0,23.0,...,292.241245,291.825941,-127.628674,-0.000237,14767.000000,15122.00000,10260.500000,-0.180062,0.011877,12683.000000
6091,142.8320,1.110475e+06,1.461151e+06,3.700340e+07,1292.0,1632.0,58446.050000,19.0,25.0,24.0,...,284.024586,283.047618,-149.094364,-0.000276,15345.000000,13081.00000,10182.000000,-0.202256,-0.079645,13125.500000
5142,83.9290,4.119381e+05,6.504286e+05,9.779356e+07,1235.0,1755.0,21680.953488,19.0,30.0,27.0,...,294.744269,294.222580,-17.828420,-0.005129,13728.228469,14225.32001,9809.053484,-0.167884,0.020972,11166.621376
1379,235.9240,4.132700e+05,4.884100e+05,6.760948e+07,1430.0,1495.0,18785.000000,22.0,26.0,23.0,...,297.733475,297.446555,-219.593532,-0.000502,13647.500000,14146.50000,10294.000000,-0.140071,0.017954,11528.000000


In [22]:
test_imp = mf.transform(test_df)        # impute test set as well

/Users/rohan/Desktop/EY-Data-And-AI-Challenge/EY-AI-And-Data-Challenge/venv/lib/python3.12/site-packages/missforest/missforest.py:490: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "
/Users/rohan/Desktop/EY-Data-And-AI-Challenge/EY-AI-And-Data-Challenge/venv/lib/python3.12/site-packages/missforest/missforest.py:494: UserWarning: In version 4.2.3, estimator fitting process is moved to `fit` method. `MissForest` will now imputes unseen missing values with fitted estimators with `transform` method. To retain the old behaviour, use `fit_transform` to fit the whole unseen data instead.
  warnings.warn(f"In version {VERSION}, estimator fitting process "
100%|██████████| 4/4 [00:00<00:00, 46.86it/s]


In [26]:
train_imp.isnull().any().sum()

np.int64(0)

Perfect, no missing values.

In [27]:
# Define X_train, X_test & y_train, y_test

X_train = train_imp.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
y_train = train_imp[['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

X_test = test_imp.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
y_test = test_imp[['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

## **Feature Selection** 

In [ ]:
# Conduct Feature selection using RFECV

estimator = RandomForestRegressor(n_estimators=100, n_jobs=-1)
rfe = RFECV(estimator=estimator, cv=10, n_jobs=-1, verbose=1)

outs = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
results = {}

for target in outs:
    rfe.fit(X_train, y_train[target])
    results[target] = X_train.columns[rfe.support_]


Fitting estimator with 30 features.
Fitting estimator with 30 features.
Fitting estimator with 30 features.
Fitting estimator with 30 features.
Fitting estimator with 30 features.
Fitting estimator with 30 features.
Fitting estimator with 30 features.
Fitting estimator with 30 features.
Fitting estimator with 29 features.
Fitting estimator with 29 features.
Fitting estimator with 29 features.
Fitting estimator with 29 features.
Fitting estimator with 29 features.
Fitting estimator with 29 features.
Fitting estimator with 29 features.
Fitting estimator with 29 features.
Fitting estimator with 28 features.
Fitting estimator with 28 features.
Fitting estimator with 28 features.
Fitting estimator with 28 features.
Fitting estimator with 28 features.
Fitting estimator with 28 features.
Fitting estimator with 28 features.
Fitting estimator with 28 features.
Fitting estimator with 27 features.
Fitting estimator with 27 features.
Fitting estimator with 27 features.
Fitting estimator with 27 fe

ValueError: All arrays must be of the same length

In [39]:
results

{'Total Alkalinity': Index(['flow_acc_phosphorous_interaction', 'flow_acc_clay_interaction',
        'phosphorous_pH_interaction', 'cec_pH_interaction', 'flow_accumulation',
        'phosphorous', 'cec_clay_ratio', 'pet', 'elevation',
        'Land Surface Temperature', 'NDVI', 'volumetric_soil_water',
        'total_evaporation_sum', 'MNDWI', 'swir22'],
       dtype='object'),
 'Electrical Conductance': Index(['flow_acc_phosphorous_interaction', 'flow_acc_clay_interaction',
        'phosphorous_pH_interaction', 'cec_pH_interaction', 'phosphorous',
        'clay', 'cec', 'pH', 'cec_clay_ratio', 'pet', 'elevation',
        'Land Surface Temperature', 'NDVI', 'volumetric_soil_water',
        'total_evaporation_sum', 'MNDWI', 'swir22'],
       dtype='object'),
 'Dissolved Reactive Phosphorus': Index(['flow_acc_phosphorous_interaction', 'flow_acc_clay_interaction',
        'phosphorous_pH_interaction', 'cec_pH_interaction', 'flow_accumulation',
        'phosphorous', 'cec', 'pH', 'cec_clay

Looks like RFECV deemed the above features as most important for the three target variables!

In [42]:
# choose features based on results of RFECV. Taking all features identified

final_feats = ['flow_acc_phosphorous_interaction', 'flow_acc_clay_interaction', 'phosphorous_pH_interaction',
               'cec_pH_interaction', 'flow_accumulation', 'phosphorous', 'cec_clay_ratio', 'pet', 'elevation',
               'Land Surface Temperature', 'NDVI', 'volumetric_soil_water', 'total_evaporation_sum', 'MNDWI',
                'swir22', 'clay', 'cec', 'pH', 'EVI', 'soil_temperature', 'evaporation_precipitation_ratio', 
                'nir', 'green', 'NDMI']

len(final_feats)

24

In [44]:
# Update X_train and X_test.

X_train_rfe = train_imp[final_feats]
X_test_rfe = test_imp[final_feats]

X_train_rfe.shape

(6523, 24)